In [ ]:

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm


plt.rcParams["figure.dpi"] = 300

import sys
from pathlib import Path

PROJECT_ROOT = next(d for d in (Path.cwd(), *Path.cwd().parents)
                    if (d / "config.yaml").exists())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import embedding_name, load_config
from src.geo import haversine_distance, heading_difference

CFG = load_config(PROJECT_ROOT)

THRESHOLDS = CFG["retrieval"]["thresholds"]
METHOD = CFG["vpr"]["method"]
RETRIEVAL_METHOD = CFG["retrieval"]["method"]
ADAPTER = CFG["vpr"].get("adapter", "none")
RESULT_DIR = PROJECT_ROOT / "results" 
EMBEDDING_DIR = PROJECT_ROOT / "data" / "embeddings" / METHOD
RETRIEVAL_DIR = RESULT_DIR / "retrieval"


EMBEDDING_NAME = embedding_name(CFG)


embedding_path = EMBEDDING_DIR / f"{EMBEDDING_NAME}_embeddings.npy"
metadata_path = EMBEDDING_DIR / f"{EMBEDDING_NAME}_metadata.parquet"

RETRIEVAL_DIR.mkdir(parents = True, exist_ok = True)

embedding_metadata = pd.read_parquet(metadata_path)
database_mask = (embedding_metadata["split"] == "database").to_numpy()
query_mask = (embedding_metadata["split"] == "query").to_numpy()
database_metadata = embedding_metadata[database_mask].reset_index(drop=True)
query_metadata = embedding_metadata[query_mask].reset_index(drop=True)

embeddings = np.load(embedding_path)

database_embeddings = embeddings[database_mask]
query_embeddings = embeddings[query_mask]

from src.run_guard import embedding_fingerprint, require_fingerprint, print_run_header, validate_config

validate_config(CFG)
print_run_header(CFG, "07_evaluation")

FINGERPRINT = embedding_fingerprint(CFG, METHOD, ADAPTER, embedding_metadata)

retrieval_path = RETRIEVAL_DIR / METHOD / f"{EMBEDDING_NAME}_retrieval.npz"

require_fingerprint(embedding_path, FINGERPRINT, what="Embeddings")
require_fingerprint(retrieval_path, FINGERPRINT, what="Retrieval-Ergebnis")

retrieval = np.load(retrieval_path)
retrieved_indices = retrieval["indices"]
similarities = retrieval["similarities"]


# Der Fingerabdruck oben deckt die config ab, nicht den Dateiinhalt. Diese
# Probe rechnet gespeicherte Similarities nach und schlaegt an, wenn 06
# nach einer Aenderung an den Embeddings nicht neu gelaufen ist.

_probe = np.random.default_rng(0).choice(
    len(query_embeddings), min(256, len(query_embeddings)), replace=False
)
_expected = (
    query_embeddings[_probe] * database_embeddings[retrieved_indices[_probe, 0]]
).sum(axis=1)

if not np.allclose(_expected, similarities[_probe, 0], atol=1e-4):
    raise RuntimeError(
        f"{retrieval_path.name} passt nicht zu {embedding_path.name}.\n"
        f"  gespeicherte Similarity (Mittel): {similarities[_probe, 0].mean():.4f}\n"
        f"  aus den Embeddings gerechnet:     {_expected.mean():.4f}\n"
        f"  -> 06_retrieval mit frischem Kernel neu ausfuehren."
    )

print(f"Retrieval geprueft: {retrieval_path.name} passt zu {embedding_path.name}")


In [ ]:
def ground_truth(query_index, query_metadata, database_metadata, threshold = 25.0):
    query = query_metadata.iloc[query_index]

    haversine_distances = haversine_distance(
        query["lat"],
        query["lon"],
        database_metadata["lat"].to_numpy(),
        database_metadata["lon"].to_numpy()
    )
    ground = np.where(haversine_distances <= threshold)[0]

    return ground, haversine_distances

In [ ]:
def recall_k( retrieved_indices, ground):

    return int(np.isin(retrieved_indices, ground).any())

In [ ]:
K_VALUES = CFG["retrieval"]["k_values"]

K_MAX = retrieved_indices.shape[1]
assert max(K_VALUES) <= K_MAX, (
    f"Nur {K_MAX} Treffer gespeichert, Recall@{max(K_VALUES)} unmoeglich"
)

db_lat = database_metadata["lat"].to_numpy()
db_lon = database_metadata["lon"].to_numpy()
db_creator = database_metadata["creator_id"].to_numpy()
db_time = database_metadata["captured_at"].to_numpy().astype("int64")

q_lat = query_metadata["lat"].to_numpy()
q_lon = query_metadata["lon"].to_numpy()
q_pano = query_metadata["is_pano"].to_numpy().astype(bool)
q_creator = query_metadata["creator_id"].to_numpy()
q_time = query_metadata["captured_at"].to_numpy().astype("int64")

db_heading = database_metadata["compass_angle"].to_numpy().astype("float64")
q_heading = query_metadata["compass_angle"].to_numpy().astype("float64")
MAX_HEADING_DIFF = float(CFG["vpr"]["max_heading_diff_deg"])

MIN_DAYS_APART = float(CFG["retrieval"]["min_days_apart"])


befunde = {}

# Fuer die Zufallsbasis: dieselbe Rechnung mit blind gezogenen statt
# gefundenen Datenbankbildern. Sagt, was Raten erreicht.
rng = np.random.default_rng(int(CFG["vpr"]["split_seed"]))


def evaluate(label, query_filter=None, gt_filter=None, block=256):
    """
    query_filter : bool-Array ueber Queries -- welche Queries zaehlen mit
    gt_filter    : Funktion(qi_block) -> bool-Matrix (len(block), n_database),
                   zusaetzliche Bedingung dafuer, dass ein DB-Bild zaehlt

    Blockweise statt je Query: die Distanzmatrix eines Blocks entsteht in
    einem numpy-Aufruf, was die Auswertung um Groessenordnungen beschleunigt.
    Bei 256 Queries x 48k Datenbankbildern sind das rund 100 MB je Block.
    """
    auswahl = (
        np.flatnonzero(query_filter)
        if query_filter is not None
        else np.arange(len(query_metadata))
    )

    n_localizable = {t: 0 for t in THRESHOLDS}
    hits = {(t, k): 0 for t in THRESHOLDS for k in K_VALUES}
    zufall = {(t, k): 0 for t in THRESHOLDS for k in K_VALUES}

    for start in tqdm(range(0, len(auswahl), block), desc=label, leave=False):
        qi = auswahl[start : start + block]

        d = haversine_distance(
            q_lat[qi, None], q_lon[qi, None], db_lat[None, :], db_lon[None, :]
        )
        if gt_filter is not None:
            # Ausgeschlossene Treffer auf unendlich setzen: sie fallen damit
            # aus jeder Schwelle heraus, ohne dass eine zweite Maske noetig ist.
            d = np.where(gt_filter(qi), d, np.inf)

        d_top = np.take_along_axis(d, retrieved_indices[qi], axis=1)
        d_zufall = np.take_along_axis(
            d, rng.integers(0, d.shape[1], size=(len(qi), max(K_VALUES))), axis=1
        )

        for t in THRESHOLDS:
            loesbar = (d <= t).any(axis=1)
            n_localizable[t] += int(loesbar.sum())
            for k in K_VALUES:
                hits[(t, k)] += int(((d_top[:, :k] <= t).any(axis=1) & loesbar).sum())
                zufall[(t, k)] += int(((d_zufall[:, :k] <= t).any(axis=1) & loesbar).sum())

    n_queries = len(auswahl)
    befunde[label] = {
        "n_queries": n_queries,
        "schwellen": {
            str(t): {
                "loesbar": n_localizable[t],
                "recall": {
                    str(k): (hits[(t, k)] / n_localizable[t] if n_localizable[t] else None)
                    for k in K_VALUES
                },
                "zufall": {
                    str(k): (zufall[(t, k)] / n_localizable[t] if n_localizable[t] else None)
                    for k in K_VALUES
                },
            }
            for t in THRESHOLDS
        },
    }

    print(f"\n{label}   (Queries: {n_queries:,})")
    print(
        f"{'Schwelle':>10} {'loesbar':>10} {'Anteil':>8} "
        + " ".join(f"R@{k:<5}" for k in K_VALUES)
    )
    for t in THRESHOLDS:
        n = n_localizable[t]
        frac = n / n_queries * 100 if n_queries else 0.0
        vals = " ".join(
            f"{hits[(t, k)] / n:<7.3f}" if n else f"{'-':<7}" for k in K_VALUES
        )
        print(f"{t:>8} m {n:>10,} {frac:>7.1f}% {vals}")
    z = " ".join(
        f"{zufall[(t, k)] / n_localizable[t]:<7.4f}" if n_localizable[t] else f"{'-':<7}"
        for t in (THRESHOLDS[len(THRESHOLDS) // 2],) for k in K_VALUES
    )
    print(f"{'Zufall':>10} {'':>10} {'':>8} {z}   (bei {THRESHOLDS[len(THRESHOLDS) // 2]} m)")


# --- 1. Standard --------------------------------------------------------
evaluate("Alle Queries")

# --- 2. Ohne Panorama-Queries (4.1) -------------------------------------
if q_pano.any():
    evaluate("Nur Nicht-Panorama-Queries", query_filter=~q_pano)
else:
    print("\nKeine Panorama-Queries im Datensatz -- Aufteilung entfaellt.")


# --- 3. Zeit-/Creator-disjunkt (4.3) ------------------------------------
def disjoint(qi):
    dt_days = np.abs(q_time[qi][:, None] - db_time[None, :]) / 86_400_000.0
    return (db_creator[None, :] != q_creator[qi][:, None]) | (dt_days > MIN_DAYS_APART)


evaluate(
    f"Hard: anderer creator_id ODER > {MIN_DAYS_APART:g} Tage Abstand",
    gt_filter=disjoint,
)


# --- 4. Blickrichtung ---------------------------------------------------
# Zwei Bilder 5 m auseinander, die in entgegengesetzte Richtungen schauen,
# haben keinen gemeinsamen Bildinhalt -- geometrisch "richtig", visuell
# unmoeglich. Ein Treffer zaehlt hier nur, wenn auch die Blickrichtung
# passt. Faellt "loesbar" gegenueber der Standardauswertung, sind das die
# Anfragen, die kein Encoder loesen kann. Steigt R@1, waren die Fehlgriffe
# dort in Wahrheit Blickrichtungsfehler.
def heading_ok(qi):
    return heading_difference(q_heading[qi][:, None], db_heading[None, :]) <= MAX_HEADING_DIFF


evaluate(
    f"Blickrichtung: Treffer nur bei <= {MAX_HEADING_DIFF:g} Grad Abweichung",
    gt_filter=heading_ok,
)


# ----------------------------------------------------------------------
# Ergebnis sichern, damit compare.py daraus die Vergleichstabelle baut
# ----------------------------------------------------------------------

import json

EVAL_PATH = RESULT_DIR / "evaluation" / f"{EMBEDDING_NAME}.json"
EVAL_PATH.parent.mkdir(parents=True, exist_ok=True)
EVAL_PATH.write_text(
    json.dumps(
        {
            "datum": pd.Timestamp.now().strftime("%Y-%m-%d"),
            "method": METHOD,
            "adapter": ADAPTER,
            "embedding_name": EMBEDDING_NAME,
            "dim": int(embeddings.shape[1]),
            "n_database": int(len(database_metadata)),
            "auswertungen": befunde,
        },
        indent=2,
    )
)
print(f"\nAuswertung gespeichert: {EVAL_PATH}")